# 03 — Stochastic Hodgkin-Huxley

Channel noise from stochastic single-particle gating (binomial tau-leaping).
Effect of channel count on noise and convergence to the deterministic limit.

In [1]:
import sys; sys.path.insert(0, '/workspace')
import os, warnings; warnings.filterwarnings('ignore')
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams['font.family'] = ['Liberation Sans','Arimo','DejaVu Sans']
matplotlib.rcParams['svg.fonttype'] = 'none'
FIG_DIR = '/mnt/results/hh_simulator/figures'
os.makedirs(FIG_DIR, exist_ok=True)
from hh_simulator import (NaChannel, KChannel, LeakChannel, PointCell,
    Simulator, step_pulse, analysis, viz)
cell = PointCell([NaChannel('classic'), KChannel('classic'), LeakChannel()])
sim = Simulator(cell)
t_eval = np.linspace(0, 40, 4001)
I_fn = step_pulse((0,40), 10.0, onset=5, dur=30)
sol_det = sim.run((0,40), I_inj=I_fn, mode='deterministic', t_eval=t_eval)

In [2]:
rng = np.random.default_rng(42)
sol_st = sim.run((0,40), I_inj=I_fn, mode='stochastic', dt=0.01,
                 N_channels={'Na':1000,'K':300}, rng=rng, record_every=5)
fig, ax = plt.subplots(figsize=(7,3.2))
ax.plot(sol_det.t, sol_det.V, 'k-', lw=1, label='deterministic')
ax.plot(sol_st.t, sol_st.V, '#0279EE', lw=0.8, alpha=0.7, label='stochastic (N=1000)')
ax.set_xlabel('Time (ms)'); ax.set_ylabel('Voltage (mV)')
ax.set_title('Channel noise: deterministic vs stochastic'); ax.legend(frameon=False)
fig.savefig(f'{FIG_DIR}/03_stochastic_vs_det.svg', bbox_inches='tight')
fig.savefig(f'{FIG_DIR}/03_stochastic_vs_det.png', bbox_inches='tight', dpi=150)
plt.show()

In [3]:
fig, axes = plt.subplots(3, 1, figsize=(7,6), sharex=True)
for ax, p, lab in zip(axes, ('m','h','n'),
    ('m (Na activation)','h (Na inactivation)','n (K activation)')):
    ax.plot(sol_st.t, sol_st.gating[p], '#0279EE', lw=0.8)
    ax.plot(sol_det.t, sol_det.gating[p], 'k--', lw=1, label='deterministic')
    ax.set_ylabel(lab); ax.legend(frameon=False, fontsize=8)
axes[-1].set_xlabel('Time (ms)')
fig.suptitle('Stochastic gating fractions (N_Na=1000)')
fig.tight_layout()
fig.savefig(f'{FIG_DIR}/03_gating_stochastic.svg', bbox_inches='tight')
fig.savefig(f'{FIG_DIR}/03_gating_stochastic.png', bbox_inches='tight', dpi=150)
plt.show()

In [4]:
Ns = [100, 300, 1000, 3000, 10000, 30000]
variances = []
for N in Ns:
    rng = np.random.default_rng(123)
    s = sim.run((0,60), I_inj=0.0, mode='stochastic', dt=0.01,
                N_channels={'Na':N,'K':N//3}, rng=rng, record_every=5)
    m = s.t > 20
    variances.append(np.var(s.V[m]))
fig, ax = plt.subplots(figsize=(5,3.5))
ax.loglog(Ns, variances, 'o-', color='#0279EE', lw=1.5)
ax.loglog(Ns, [variances[0]*(Ns[0]/n) for n in Ns], 'k--', lw=1, label='$\\propto 1/N$')
ax.set_xlabel('N (Na channels)'); ax.set_ylabel('Var(V) at rest (mV$^2$)')
ax.set_title('Channel-noise variance scaling'); ax.legend(frameon=False)
fig.savefig(f'{FIG_DIR}/03_noise_scaling.svg', bbox_inches='tight')
fig.savefig(f'{FIG_DIR}/03_noise_scaling.png', bbox_inches='tight', dpi=150)
plt.show()